In [1]:
import pandas as pd
import numpy as np

In [4]:
import pandas as pd
df = pd.read_csv('/content/train.csv', on_bad_lines='skip', engine='python')

In [5]:
df.isnull().sum()

,0
id,0
qid1,0
qid2,0
question1,0
question2,2
is_duplicate,0


In [7]:
new=df.sample(30000)

In [8]:
new.isnull().sum()
new.duplicated().sum()

np.int64(0)

In [9]:
ques=new[['question1','question2']]
ques.head()

,question1,question2
113396,Was it actually possible to keep Hiroshi Ouchi...,Is it possible to lose belly fat after age 35?
175910,What are the best applications of the best pro...,What are the best programming languages to lea...
153478,What is an isotherm?,What is adiabatic elasticity and isothermal el...
85139,Which post should I prefer to pursue in AFCAT ...,Will not having enough money / credit to purch...
60633,How do you deal with Monday morning blues?,How do you beat the Monday blues?


In [11]:
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

# Fill NaN values with empty strings before merging texts
questions = list(ques['question1'].fillna('')) + list(ques['question2'].fillna(''))

cv = CountVectorizer(max_features=3000)
q1_arr, q2_arr = np.vsplit(cv.fit_transform(questions).toarray(),2)

In [14]:
temp_df1 = pd.DataFrame(q1_arr, index= ques.index)
temp_df2 = pd.DataFrame(q2_arr, index= ques.index)
temp_df = pd.concat([temp_df1, temp_df2], axis=1)
temp_df.shape

(30000, 6000)

In [18]:
temp_df['is_duplicate'] = new['is_duplicate']

In [22]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(temp_df.iloc[:,0:-1].values,temp_df.iloc[:,-1].values,test_size=0.2,random_state=1)

In [25]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
rf=RandomForestClassifier()
rf.fit(X_train,y_train)
y_pred=rf.predict(X_test)
accuracy_score(y_test,y_pred)

0.7461666666666666

In [26]:
from xgboost import XGBClassifier
xgb = XGBClassifier()
xgb.fit(X_train,y_train)
y_pred = xgb.predict(X_test)
accuracy_score(y_test,y_pred)

0.7315